In [1]:
import pandas as pd
import gseapy as gp

hub_genes = pd.read_csv(r"C:\tri_disease_project\results\tables\hub_genes_final.csv")['gene_symbol'].tolist()
shared_genes = pd.read_csv(r"C:\tri_disease_project\data\processed\shared_genes_protein_coding.csv")['hgnc_symbol'].tolist()

print("Hub genes:", len(hub_genes))
print("Shared genes:", len(shared_genes))

regulatory_gene_sets = ['TRRUST_Transcription_Factors_2019', 'miRTarBase_2017']

def run_regulatory_enrichment(gene_list, label, outdir):
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=regulatory_gene_sets,
        organism='human',
        outdir=outdir,
        cutoff=0.05
    )
    results = enr.results
    results['gene_set_input'] = label
    return results

reg_results_hub = run_regulatory_enrichment(hub_genes, 'hub_17', r"C:\tri_disease_project\results\tables\enrichr_regulatory_hub")
reg_results_shared = run_regulatory_enrichment(shared_genes, 'shared_690', r"C:\tri_disease_project\results\tables\enrichr_regulatory_shared")

print("\n=== Top TF/miRNA regulators of the 17 HUB genes ===")
print(reg_results_hub.sort_values('Adjusted P-value').head(20)[['Gene_set', 'Term', 'Overlap', 'Adjusted P-value', 'Genes']].to_string(index=False))

print("\n=== Top TF/miRNA regulators of the 690 SHARED genes ===")
print(reg_results_shared.sort_values('Adjusted P-value').head(20)[['Gene_set', 'Term', 'Overlap', 'Adjusted P-value', 'Genes']].to_string(index=False))

reg_results_hub.to_csv(r"C:\tri_disease_project\results\tables\regulatory_hub_full.csv", index=False)
reg_results_shared.to_csv(r"C:\tri_disease_project\results\tables\regulatory_shared_full.csv", index=False)

Hub genes: 17
Shared genes: 690

=== Top TF/miRNA regulators of the 17 HUB genes ===
                         Gene_set         Term Overlap  Adjusted P-value                                     Genes
TRRUST_Transcription_Factors_2019  STAT3 mouse    6/79      3.346231e-09            IL6;CCND1;PTEN;BCL2;PIK3R1;TNF
TRRUST_Transcription_Factors_2019   TP53 human   7/164      3.346231e-09  SMAD3;CCND1;PTEN;BCL2;ESR1;TP53;PPARGC1A
TRRUST_Transcription_Factors_2019    SP1 human   9/472      3.346231e-09 IL6;CCND1;LEP;PTEN;BCL2;LPL;APOE;TNF;ESR1
TRRUST_Transcription_Factors_2019   EGR1 human    6/88      4.212082e-09             IL6;CCND1;PTEN;PPARG;TNF;TP53
TRRUST_Transcription_Factors_2019  NFKB1 mouse   7/200      7.461322e-09     IL6;CCND1;PTEN;BCL2;APOE;TNF;PPARGC1A
TRRUST_Transcription_Factors_2019  FOXO1 mouse    5/49      1.674164e-08           CCND1;LPL;PPARG;PIK3R1;PPARGC1A
TRRUST_Transcription_Factors_2019   RELA mouse   6/121      1.680571e-08              IL6;CCND1;PTEN;BCL2;APOE

In [2]:
def filter_human_only(df):
    # TRRUST: keep only rows where Term ends in "human"; miRTarBase: keep only hsa- (human) miRNAs
    trrust_human = df[(df['Gene_set'] == 'TRRUST_Transcription_Factors_2019') & (df['Term'].str.endswith('human'))]
    mirna_human = df[(df['Gene_set'] == 'miRTarBase_2017') & (df['Term'].str.startswith('hsa-'))]
    return pd.concat([trrust_human, mirna_human]).sort_values('Adjusted P-value')

hub_human = filter_human_only(reg_results_hub)
shared_human = filter_human_only(reg_results_shared)

print("=== Top human TF/miRNA regulators — HUB genes ===")
print(hub_human.head(15)[['Gene_set', 'Term', 'Overlap', 'Adjusted P-value', 'Genes']].to_string(index=False))

print("\n=== Top human TF/miRNA regulators — SHARED genes ===")
print(shared_human.head(15)[['Gene_set', 'Term', 'Overlap', 'Adjusted P-value', 'Genes']].to_string(index=False))

hub_human.to_csv(r"C:\tri_disease_project\results\tables\regulatory_hub_human_only.csv", index=False)
shared_human.to_csv(r"C:\tri_disease_project\results\tables\regulatory_shared_human_only.csv", index=False)

=== Top human TF/miRNA regulators — HUB genes ===
                         Gene_set         Term Overlap  Adjusted P-value                                     Genes
TRRUST_Transcription_Factors_2019   TP53 human   7/164      3.346231e-09  SMAD3;CCND1;PTEN;BCL2;ESR1;TP53;PPARGC1A
TRRUST_Transcription_Factors_2019    SP1 human   9/472      3.346231e-09 IL6;CCND1;LEP;PTEN;BCL2;LPL;APOE;TNF;ESR1
TRRUST_Transcription_Factors_2019   EGR1 human    6/88      4.212082e-09             IL6;CCND1;PTEN;PPARG;TNF;TP53
TRRUST_Transcription_Factors_2019  STAT3 human   6/142      3.875821e-08              IL6;CCND1;LEP;PTEN;BCL2;TP53
TRRUST_Transcription_Factors_2019   RELA human   7/301      6.772101e-08          IL6;CCND1;PTEN;BCL2;LPL;TNF;TP53
TRRUST_Transcription_Factors_2019  NFKB1 human   7/303      6.772101e-08          IL6;CCND1;PTEN;BCL2;LPL;TNF;TP53
TRRUST_Transcription_Factors_2019  NCOR1 human     3/7      3.443751e-07                           PPARG;IGF1;ESR1
TRRUST_Transcription_Factors_2